# 01 - Análise exploratória: entregas urbanas de moto

**Objetivo:** conhecer a base simulada de entregas antes de qualquer limpeza ou modelagem.

**Perguntas de negócio que guiam esta etapa**
1. Qual o volume de entregas e como ele se distribui por status?
2. Distância, tempo de trânsito e valor estão em faixas plausíveis para moto em área urbana?
3. O horário de pico de trânsito muda tempo, atraso e cancelamento?
4. Quais bairros concentram a origem (coleta) e o destino das entregas?

**Fonte:** `data/raw/dados_entregas_simulados.csv`, gerado por `src/logistica_rotas/ingestao/simulacao.py` (cidade e bairros fictícios, `random_state=42`).

> Por que simular? Permite controlar as regras de negócio (prazo, velocidade, pico) e saber de antemão o que a análise deveria encontrar - bom para treinar e validar o método.

## 1. Configuração

Localiza a raiz do projeto e coloca `src/` no caminho de importação, para usar as funções do pacote `logistica_rotas`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Sobe pastas até achar o CLAUDE.md (raiz do projeto), funcione o notebook de onde for aberto.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
sys.path.insert(0, str(RAIZ / "src"))

from logistica_rotas.ingestao.simulacao import CAMINHO_PADRAO, FUSO, gerar_e_salvar

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 20)
print("Raiz do projeto:", RAIZ)

## 2. Fonte de dados

Se o CSV ainda não existir (ele não vai para o Git), é gerado agora. Mesma seed = mesmos dados sempre.

In [ ]:
if not CAMINHO_PADRAO.exists():
    gerar_e_salvar()
    print("Dataset gerado.")

df = pd.read_csv(CAMINHO_PADRAO, encoding="utf-8")
# Converte para datetime com fuso explícito (diretriz do projeto).
df["data_hora_pedido"] = pd.to_datetime(df["data_hora_pedido"], utc=True).dt.tz_convert(FUSO)
for col in ["bairro_origem", "bairro_destino", "horario_pico", "status"]:
    df[col] = df[col].astype("category")

print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas".replace(",", "."))
df.head()

## 3. Estrutura e tipos

Confere se cada coluna foi lida com o tipo certo. Unidades estão no nome: `_km`, `_min`, `_brl`.

In [ ]:
df.info()

## 4. Checklist de validação

Itens do checklist da skill `logistica-dados`. Aqui só **identificamos** problemas; o tratamento fica para o notebook de limpeza.

In [ ]:
concluidas = df["status"] != "cancelado"
velocidade_kmh = df["distancia_km"] / (df["tempo_transito_min"] / 60)

checagens = pd.Series({
    "id_pedido duplicado": df["id_pedido"].duplicated().sum(),
    "nulos em tempo_transito_min (total)": df["tempo_transito_min"].isna().sum(),
    "nulos em tempo_transito_min (cancelados)": df.loc[~concluidas, "tempo_transito_min"].isna().sum(),
    "nulos em tempo_transito_min (não cancelados)": df.loc[concluidas, "tempo_transito_min"].isna().sum(),
    "distância <= 0": (df["distancia_km"] <= 0).sum(),
    "tempo <= 0": (df["tempo_transito_min"] <= 0).sum(),
    "velocidade média < 3 km/h (suspeito)": (velocidade_kmh < 3).sum(),
    "velocidade média > 60 km/h (impossível p/ moto urbana)": (velocidade_kmh > 60).sum(),
}, name="quantidade")
checagens.to_frame()

**Leitura:** os nulos de tempo devem coincidir com os cancelados (entrega não aconteceu). Velocidades médias muito baixas indicam erro de registro do tempo - candidatos a outlier.

## 5. Estatísticas descritivas - variáveis numéricas

In [ ]:
numericas = ["distancia_km", "tempo_transito_min", "prazo_prometido_min", "valor_entrega_brl"]
desc = df[numericas].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
desc["cv_%"] = desc["std"] / desc["mean"] * 100  # coeficiente de variação
desc

**Como ler:** compare média e mediana (50%). Média bem acima da mediana indica cauda longa à direita - o máximo do tempo de trânsito, muito acima do percentil 99, é sinal de outlier.

## 6. Variáveis categóricas

In [ ]:
status = pd.DataFrame({
    "qtd": df["status"].value_counts(),
    "%": df["status"].value_counts(normalize=True) * 100,
})
status

In [ ]:
pd.crosstab(df["horario_pico"], df["status"], normalize="index", margins=True) * 100

In [ ]:
bairros = pd.DataFrame({
    "origens_%": df["bairro_origem"].value_counts(normalize=True) * 100,
    "destinos_%": df["bairro_destino"].value_counts(normalize=True) * 100,
}).sort_values("origens_%", ascending=False)
bairros

## 7. Primeiros indicadores

Definições (da skill `logistica-dados`; registrar em `docs/` antes de virar KPI oficial):
- **OTD** = entregas no prazo / entregas concluídas. Cancelados ficam fora do denominador.
- **Taxa de cancelamento** = cancelados / total de pedidos.
- Para tempo, usamos a **mediana**, que é menos sensível aos outliers vistos acima.

In [ ]:
def resumo_operacional(grupo: pd.DataFrame) -> pd.Series:
    """Indicadores básicos de um grupo de entregas."""
    feitas = grupo[grupo["status"] != "cancelado"]
    return pd.Series({
        "pedidos": len(grupo),
        "otd_%": (feitas["status"] == "entregue").mean() * 100,
        "cancelamento_%": (grupo["status"] == "cancelado").mean() * 100,
        "distancia_media_km": grupo["distancia_km"].mean(),
        "tempo_mediano_min": feitas["tempo_transito_min"].median(),
        "valor_medio_brl": grupo["valor_entrega_brl"].mean(),
    })

geral = resumo_operacional(df).rename("geral")
por_pico = df.groupby("horario_pico", observed=True).apply(resumo_operacional, include_groups=False)
pd.concat([geral.to_frame().T, por_pico])

In [ ]:
df["hora"] = df["data_hora_pedido"].dt.hour
df["dia_semana"] = df["data_hora_pedido"].dt.day_name()
por_hora = df.groupby("hora").apply(resumo_operacional, include_groups=False)
por_hora[["pedidos", "otd_%", "tempo_mediano_min"]]

## 8. Achados preliminares e próximos passos

**Achados (preencher após rodar e conferir os números):**
- Volume e período da base.
- OTD geral e diferença entre pico e fora do pico.
- Horários de maior demanda (almoço e jantar) x horários de pico de trânsito - não são os mesmos.
- Outliers de tempo de trânsito encontrados no checklist.

**Limitações:**
- Dados simulados: as relações encontradas refletem as regras do gerador, não uma operação real.
- Distância é linha reta x fator de tortuosidade (1,3), não a distância real pela malha viária.

**Próximos passos:**
1. `02_limpeza`: tratar outliers de tempo e registrar a regra em `docs/dicionario_dados.md`.
2. Mover `resumo_operacional` para `src/logistica_rotas/metricas/` com teste em `tests/`.
3. Gráficos: distribuição do tempo de trânsito, OTD por hora e por bairro de destino.